In [ ]:
import pandas as pd
import matplotlib as plt
import numpy as np

df = pd.read_csv(
    r'C:\Users\tomas\New folder\Data Preprocessing\globalmart_dirty_orders_2000.csv', 
    keep_default_na=False,
    na_values=['-', '#N/A', 'N/A', 'NULL', '?', 'unknown', 'NONE', 'na', 'None', 'none', 'NaN', 'NA', 'nan'] # catch all null values before parsing

)   

df.index = df.index + 2 ## shifted index to align with csv

columns = [
    'record_id', 'source_system', 'customer_id', 'customer_name', 
    'customer_email', 'phone_raw', 'age_raw', 'gender_raw', 
    'signup_date_raw', 'order_id', 'order_date_raw', 'ship_date_raw', 
    'delivery_date_raw', 'customer_timezone', 'country_raw', 'city_raw', 
    'postal_code_raw', 'product_category_raw', 'product_name_raw', 
    'sku_raw', 'quantity_raw', 'unit_price_raw', 'currency_raw', 
    'discount_raw', 'shipping_cost_raw', 'item_weight_raw', 
    'payment_method_raw', 'order_status_raw', 'returned_flag_raw', 
    'return_reason_raw', 'satisfaction_score_raw', 'loyalty_points_raw', 
    'site_visits_last30_raw', 'support_tickets_90d_raw', 
    'distance_to_warehouse_km_raw', 'campaign_source_raw', 
    'customer_segment_raw', 'support_ticket_date_raw', 
    'complaint_text_raw', 'ingestion_batch', 'data_source_note'
]

In [ ]:
## PART 1
def returnRows(df):
    return f"Total row count: {len(df)}"

def returnDatatype(df):
    return df.dtypes

def checkMissingValues(df, columns):
    null_rows = df[columns].isnull().sum() ## returns how many null rows are there in the series
    missing_values = null_rows[null_rows > 0] ## filters all boolean values that are greater than 0 to be passed through {missing_values} variable
    total_rows = len(df)
    
    percentages = (missing_values / total_rows) * 100
    print("--------------------Missing Rows--------------------")
    summary_df = pd.DataFrame({
        'Missing Count': missing_values,
        'Percentage (%)': percentages.round(2)  
    })

    return summary_df

## specifies which columns and row has null values
def specifyWhere(df, columns):
    missing_locations = {}
    for col in columns:
        null_indeces = df[df[col].isnull()].index.tolist()

        if null_indeces:
            missing_locations[col] = null_indeces

    return missing_locations

## if {value} of {column_name} HAS duplicates:
    ## do: 
        ## return the row where the {value} is duplicated
        ## filter all {value} to show all true
        ## show the number of duplicated {value} in a column

def checkSpecificColumnDuplicates(df, column_name):
    duplicate_mask = df.duplicated(subset=[column_name], keep=False) ## checks every value in index to check if {value} has duplicates
    duplicated_rows = df[duplicate_mask] ## passes all duplicates into duplicated_rows
    duplicated_rows = duplicated_rows.dropna(subset=[column_name]) ## drops all values containing {NaN}
    duplicates_clean_view = duplicated_rows[['record_id', column_name]]

    print(f"Total amount of duplicated rows: {len(duplicates_clean_view)}")
    return duplicates_clean_view

def genderRemap(df):
    print("unique:")
    print(f"{df["gender_raw"].unique()}\n")
    clean_source = df["gender_raw"].str.upper().str.strip()

    mapping = {
            r'(?i)^(NON-BINARY|NB)$': 'Non-Binary', 
            r'(?i)^(MALE|M)$': 'Male', 
            r'(?i)^(FEMALE|F)$':'Female', 
            r'(?i)^PREFER NOT TO SAY$': 'Prefer not to say'
            
            }
    
    print(f"dict: {mapping} \n")
    df["gender_raw"] = clean_source.replace(mapping, regex=True)
    
    ## recheck
    print("remapped gender_raw")
    return df["gender_raw"]

def sourceRemap(df):
    print("unique: ")
    print(f"{df["source_system"].unique()}\n")
    clean_source = df["source_system"].str.upper().str.strip()

    mapping = {
        r'(?i)^(STORE|IN-STORE)$': 'In-Store',
        r'(?i)^(MOBILE-APP|MOBILE APP|MOBILE|APP)$': 'Mobile App',
        r'(?i)^(CALL CENTRE|CALL CENTER)$': 'Call Center',
        r'(?i)^(PARTNER-API|PARTNER API)$': 'Partner-API',
        r'(?i)^(MARKETPLACE|MARKET PLACE)$': 'Marketplace',
        r'MARKETPLACE | MARKET PLACE': 'Marketplace',
        r'(?i)^WEB$': 'Web'
    }
    
    print(f"dict: {mapping} \n")
    df["source_system"] = clean_source.replace(mapping, regex=True)

    print("remapped source_system")
    return df["source_system"]

def countryRemap(df):
    print("unique: ")
    print(df["country_raw"].unique())
    
    clean_source = df["country_raw"].str.upper().str.strip()
    mapping = {
        r'(?i)^AUS?$|^Australia$': 'Australia',
        r'(?i)^DE$|^Ger(many)?$|^Deutschland$': 'Germany',
        r'(?i)^FR(ance|nce)?$': 'France',
        r'(?i)^SG$|^Singapore$|^S\'pore$': 'Singapore',
        r'(?i)^JP$|^Japan$|^Nippon$': 'Japan',
        r'(?i)^CA(nada|nda)?$': 'Canada',
        r'(?i)^BR(asil|azil)?$': 'Brazil',
        r'(?i)^PH(L|ils\.?)?$|^Philippines$|^Pilipinas$': 'Philippines',
        r'(?i)^US(A|\.S\.A\.)?$|^United States$|^America$': 'United States',
        r'(?i)^CH$|^Swi(tzerland|ss)$|^Suisse$': 'Switzerland'
    }
    
    print(f"dict: {mapping} \n")
    df["country_raw"] = clean_source.replace(mapping, regex=True)

    print("remapped country_raw")
    return df["country_raw"]

def detectInvalidAge(df):
    age_numeric = pd.to_numeric(df["age_raw"], errors='coerce')
    invalid_condition = (age_numeric >= 100) | (age_numeric <= 13) | (age_numeric.isna())
    invalid_rows = df[invalid_condition]
    return invalid_rows[['record_id', 'age_raw']]

def detectInvalidDiscount(df):
    if df["discount_raw"].dtype == '0':
        mapping = {
        'freebie': '100%',
        'ten percent': '10%'
        }
            
        df["discount_raw"] = df["discount_raw"].replace(mapping, regex=True)
        df["discount_raw"] = df["discount_raw"].str.rstrip('%').astype(float)
        
    discounts = pd.to_numeric(df["discount_raw"], errors='coerce')    
    invalid_condition = (discounts <= 0) | (discounts >= 101) | (discounts.isna())
    invalid_rows = df[invalid_condition]
    return invalid_rows[['record_id', 'discount_raw']]

def invalidShipOrder(df):
    order_date = pd.to_datetime(df["order_date_raw"], format='mixed', errors='coerce')
    delivery_date = pd.to_datetime(df["delivery_date_raw"], format='mixed', errors='coerce')
    invalid_condition = (order_date > delivery_date)
    invalid_rows = df[invalid_condition]
    return invalid_rows[['record_id', 'order_date_raw', 'delivery_date_raw']]

def invalidEmail(df): ## confirmed no false positives (look at mateo.nez140@example..com) ((double dots))
    regex = r'^[\w.-]+@([\w-]+\.)+[\w-]{2,4}$'
    is_valid = df['customer_email'].str.contains(regex, regex=True, na=False)
    invalid_email = df[~is_valid]
    return invalid_email[['record_id', 'customer_email']]



## email validation regex: ^[\w-\.]+@([\w-]+\.)+[\w-]{2,4}$     

In [ ]:
## PART 2 
# missing categorical contexts
def missingCustomerEmail(df):
    missing_customer_email = df[df['customer_email'].isna()]
    return missing_customer_email[['record_id', 'customer_email']]

def missingDeliveryDate(df):
    missing_delivery_date = df[df['delivery_date_raw'].isna()]
    return missing_delivery_date[['record_id', 'delivery_date_raw']]

def missingSupportTicketDate(df):
    missing_support_ticket = df[df['support_ticket_date_raw'].isna()]
    return missing_support_ticket[['record_id', 'support_ticket_date_raw']]

def missingReturnReason(df):
    missing_return_reason = df[df['return_reason_raw'].isna()]
    return missing_return_reason[['record_id', 'return_reason_raw']]


In [ ]:
## PART 3
def fixTime(df):
   date_columns = ['signup_date_raw', 'order_date_raw', 'ship_date_raw', 'delivery_date_raw', 'support_ticket_date_raw']
   for i in date_columns:
      parsed_dates = pd.to_datetime(df[i], format="mixed", errors="coerce")
      df[i] = parsed_dates.dt.strftime('%Y-%m-%d')


In [ ]:
## Clean Everything
def cleanEverything(df):
    genderRemap(df)
    sourceRemap(df)
    countryRemap(df)
    fixTime(df)

In [11]:
cleanEverything(df)
df

unique:
<StringArray>
['Non-Binary', nan, 'Male', 'Female', 'Prefer not to say', '']
Length: 6, dtype: str

dict: {'(?i)^(NON-BINARY|NB)$': 'Non-Binary', '(?i)^(MALE|M)$': 'Male', '(?i)^(FEMALE|F)$': 'Female', '(?i)^PREFER NOT TO SAY$': 'Prefer not to say'} 

remapped gender_raw
unique: 
<StringArray>
[ 'Mobile App',         'Web',    'In-Store', 'Marketplace', 'Call Center',
 'Partner-API',           nan,            '']
Length: 8, dtype: str

dict: {'(?i)^(STORE|IN-STORE)$': 'In-Store', '(?i)^(MOBILE-APP|MOBILE APP|MOBILE|APP)$': 'Mobile App', '(?i)^(CALL CENTRE|CALL CENTER)$': 'Call Center', '(?i)^(PARTNER-API|PARTNER API)$': 'Partner-API', '(?i)^(MARKETPLACE|MARKET PLACE)$': 'Marketplace', 'MARKETPLACE | MARKET PLACE': 'Marketplace', '(?i)^WEB$': 'Web'} 

remapped source_system
unique: 
<StringArray>
[    'Australia',       'Germany',             nan,        'France',
     'Singapore',         'Japan',        'Canada',        'Brazil',
   'Philippines', 'United States',        'U.S.

,record_id,source_system,customer_id,customer_name,customer_email,phone_raw,age_raw,gender_raw,signup_date_raw,order_id,...,loyalty_points_raw,site_visits_last30_raw,support_tickets_90d_raw,distance_to_warehouse_km_raw,campaign_source_raw,customer_segment_raw,support_ticket_date_raw,complaint_text_raw,ingestion_batch,data_source_note
2,GM-00135,Mobile App,CUST-000429,MarÃ­a Santos,mara.santos385@company.com,+41 31 579 05 71,39,Non-Binary,2025-12-05,ORD-4OHL5THL,...,698,500,0,1263.0,Email Campaign,occasional,2026-04-08,El producto llegÃ³ tarde.,batch_B_latin1,"Merged from checkout, CRM, and support extracts."
3,GM-00347,Web,CUST-001323,Mateo NÃºÃ±ez,mateo.nez140@example..com,+41 14 190 82 74,38,NaN,2022-02-08,ORD-9APGI32O,...,450,NaN,3,1502.33,Google Ads,Dormant,2025-09-29,Missing manual in box.,batch_B_latin1,Currency conversion not yet standardized.
4,GM-01214,Web,CUST-000818,Maria Davis,maria.davis@companycom,+41 16 396 43 87,20,Male,2022-05-09,ORD-AH4THJF1,...,976,10,3,8.7,Influencer,premium,2025-09-10,Produit arrivé cassé.,api_stream_v2,Currency conversion not yet standardized.
5,GM-00794,In-Store,CUST-000179,Chloé Kim,chlo.kim@companycom,+63 938 520 4772,67,Non-Binary,2022-11-23,ORD-GKXXNTI5,...,331,19,1,1288.92,FB,Regular,2026-04-22,Produit arrivé cassé.,batch_B_latin1,Dates came from mixed regional settings.
6,GM-00338,Marketplace,CUST-000281,Sofia García,sofia.garca421@gmail.com,NaN,29,Male,2021-04-17,ORD-7JJHNBT3,...,527,11,1,1215.43,Google Ads,NEW,2025-08-03,Received wrong color.,batch_B_latin1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1997,GM-00707,Partner-API,CUST-001235,François Lee,franois.lee40@gmai.com,+41 19 234 38 56,51,Female,2023-06-29,ORD-AIRO1TAI,...,509,17,0,1370.48,Organic,at-risk,2024-07-31,Produit arrivé cassé.,batch_C_windows1252,"Merged from checkout, CRM, and support extracts."
1998,GM-00443,Call Center,CUST-000939,Chloe Johnson,chloe.johnson680@outlook.com,+63 928 133 0943,71,Prefer not to say,2021-08-30,ORD-MG9D80A7,...,573,11,0,NaN,google,Loyal,2025-01-30,NaN,batch_B_latin1,Duplicate order records possible due to API re...
1999,GM-00159,Call Center,CUST-000356,Jay Müller,jay.mller90@company.com,,28,NaN,2021-11-21,ORD-YDB90MTX,...,634,14,1,239.62,FB,first-time,2025-02-21,NaN,api_stream_v2,Some returns are recorded only in support tick...
2000,GM-00244,Web,CUST-001339,José O’Connor,jos.oconnor768@hotmail.com,+41 96 669 42 39,23,Female,2025-03-18,ORD-5KFSNRRL,...,628,5,0,220.44,,premium,2025-07-06,NaN,api_stream_v2,Dates came from mixed regional settings.
